# Fine-tune F5-TTS tiếng Việt trên giọng CapCut

Nhân bản một trong ba giọng CapCut (`co_gai_hoat_ngon`, `nguon_nho_ngot_ngao`, `thanh_nien_tu_tin`) bằng cách fine-tune từ base **F5-TTS-Vietnamese-1000h**.

**Runtime bắt buộc:** Runtime → Change runtime type → **GPU (T4)**. Base model ~5.4 GB nên cần khoảng 25 GB dung lượng đĩa Colab — mặc định đủ.

---

### ⚠️ Ràng buộc license — đọc trước khi chạy

Base `hynt/F5-TTS-Vietnamese-ViVoice` phát hành theo **CC-BY-NC-SA-4.0: chỉ nghiên cứu, cấm dùng thương mại**. Mọi model fine-tune từ nó thừa hưởng ràng buộc đó (điều khoản ShareAlike). Nếu giọng này định dùng cho kênh kiếm tiền thì **không đi đường này** — dùng Piper + `vi_VN-vais1000` (CC-BY-4.0) thay thế:

```bash
python scripts/capcut_clone_dataset.py export-piper --voice co_gai_hoat_ngon
```

Chất lượng Piper thấp hơn nhưng license sạch và cắm thẳng vào pipeline OmniCast qua provider `piper` sẵn có.

---

### Chuẩn bị ở máy local trước

```bash
cd implementation
python scripts/capcut_clone_dataset.py export-f5 --voice co_gai_hoat_ngon
```

Lệnh này tạo `output/voice_clone/co_gai_hoat_ngon/f5_co_gai_hoat_ngon.zip` (~100 MB) gồm các cặp `NNN.wav` + `NNN.txt` ở 24 kHz — đúng định dạng `prepare_metadata.py` đọc, không phải resample.

In [ ]:
#@title 1. Kiểm tra GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), "Chưa bật GPU: Runtime > Change runtime type > GPU"
cap = torch.cuda.get_device_capability()
print(f"compute capability {cap[0]}.{cap[1]} | bf16 hỗ trợ: {torch.cuda.is_bf16_supported()}")
# T4 là kiến trúc Turing (7.5): chỉ có fp16, không có bf16. Ghi nhớ để xử lý nếu
# trainer mặc định bf16 và báo lỗi ở cell fine-tune.

In [ ]:
#@title 2. Cài F5-TTS-Vietnamese
%cd /content
!git clone -q https://github.com/nguyenthienhy/F5-TTS-Vietnamese
%cd /content/F5-TTS-Vietnamese
!pip install -q -e . 2>&1 | tail -5
print("cài xong")

In [ ]:
#@title 3. Tải base model tiếng Việt (5.4 GB)
from huggingface_hub import hf_hub_download
import shutil, pathlib

REPO = "hynt/F5-TTS-Vietnamese-ViVoice"
ckpt = hf_hub_download(REPO, "model_last.pt")
# Trong repo này `config.json` CHÍNH LÀ file vocab, không phải config —
# README của tác giả bảo đổi tên nó thành vocab.txt.
vocab = hf_hub_download(REPO, "config.json")

pathlib.Path("/content/base").mkdir(exist_ok=True)
shutil.copy(ckpt, "/content/base/pretrained_vi.pt")
shutil.copy(vocab, "/content/base/vocab.txt")

n = sum(1 for _ in open("/content/base/vocab.txt", encoding="utf-8"))
print(f"checkpoint: {pathlib.Path('/content/base/pretrained_vi.pt').stat().st_size/1e9:.1f} GB")
print(f"vocab: {n} token")

In [ ]:
#@title 4. Upload dataset zip
VOICE = "co_gai_hoat_ngon"  #@param ["co_gai_hoat_ngon", "nguon_nho_ngot_ngao", "thanh_nien_tu_tin"]

import os, glob, zipfile
os.makedirs("/content/F5-TTS-Vietnamese/data/your_dataset", exist_ok=True)

from google.colab import files
print(f"Chọn file f5_{VOICE}.zip từ máy bạn…")
uploaded = files.upload()

zip_name = next(iter(uploaded))
with zipfile.ZipFile(zip_name) as z:
    z.extractall("/content/F5-TTS-Vietnamese/data/your_dataset")

wavs = glob.glob("/content/F5-TTS-Vietnamese/data/your_dataset/*.wav")
txts = glob.glob("/content/F5-TTS-Vietnamese/data/your_dataset/*.txt")
print(f"{len(wavs)} wav, {len(txts)} txt")
assert len(wavs) == len(txts) and wavs, "wav và txt không khớp số lượng"

In [ ]:
#@title 5. Dựng metadata (stage 1 của fine_tuning.sh)
%cd /content/F5-TTS-Vietnamese
!python prepare_metadata.py
!head -3 data/your_training_dataset/metadata.csv
!ls data/your_training_dataset/wavs | wc -l

In [ ]:
#@title 6. Vocab — bỏ qua stage 2 & 3
# fine_tuning.sh có 2 stage mở rộng vocab + embedding. Chúng dành cho trường hợp
# đi từ base Emilia (Trung/Anh) sang tiếng Việt. Ta fine-tune TỪ base đã là tiếng
# Việt, nên chỉ cần vocab của base — với điều kiện corpus không đẻ ra ký tự lạ.
# Cell này kiểm chứng điều kiện đó thay vì tin suông.
import shutil, glob

vocab = set(open("/content/base/vocab.txt", encoding="utf-8").read().splitlines())
chars = set()
for path in glob.glob("data/your_dataset/*.txt"):
    chars |= set(open(path, encoding="utf-8").read())
chars -= {"\n"}

missing = sorted(c for c in chars if c not in vocab)
print(f"vocab base: {len(vocab)} token | ký tự corpus: {len(chars)} | thiếu: {len(missing)}")
if missing:
    print("THIẾU:", missing)
    raise SystemExit(
        "Corpus có ký tự ngoài vocab base. Phải chạy check_vocab_pretrained.py + "
        "extend_embedding_pretrained.py, hoặc lọc bỏ ký tự đó khỏi transcript."
    )

shutil.copy("/content/base/vocab.txt", "data/your_training_dataset/vocab.txt")
print("dùng thẳng vocab base — không cần mở rộng embedding")

In [ ]:
#@title 7. Trích xuất đặc trưng (stage 4)
!python src/f5_tts/train/datasets/prepare_csv_wavs.py \
    data/your_training_dataset data/your_training_dataset --workers 2
!ls -la data/your_training_dataset

In [ ]:
#@title 8. Khớp đường dẫn dataset mà trainer mong đợi
# F5-TTS gốc tìm dataset ở `data/{dataset_name}_{tokenizer}` (vd `..._char`),
# còn fine_tuning.sh của fork này ghi thẳng vào `data/your_training_dataset`.
# Thay vì đoán fork đã sửa hay chưa, tạo luôn alias để cả hai cách đều tìm thấy.
import os, shutil

src = "data/your_training_dataset"
alias = "data/your_training_dataset_char"
if not os.path.exists(alias):
    try:
        os.symlink(os.path.abspath(src), alias)
        print("symlink:", alias)
    except OSError:
        shutil.copytree(src, alias)
        print("copy:", alias)
!ls data/ && ls $alias 2>/dev/null | head

In [ ]:
#@title 9. Fine-tune
# Tham số đã chỉnh cho dataset nhỏ (~500 câu, ~40 phút) trên T4 16GB.
# fine_tuning.sh gốc đặt warmup 20000 update — con số cho bộ 1000 giờ, dùng ở
# đây thì model sẽ chưa kịp rời giai đoạn khởi động trước khi ta dừng train.
BATCH = 3000          # frame; nếu CUDA OOM thì hạ 1600 và đặt GRAD_ACC=2
GRAD_ACC = 1
LR = 1e-5             # cao hơn dễ mất chất tiếng Việt của base
WARMUP = 200
SAVE_EVERY = 1000
EPOCHS = 400

!python src/f5_tts/train/finetune_cli.py \
    --exp_name F5TTS_Base \
    --dataset_name your_training_dataset \
    --tokenizer char \
    --learning_rate {LR} \
    --batch_size_per_gpu {BATCH} \
    --grad_accumulation_steps {GRAD_ACC} \
    --num_warmup_updates {WARMUP} \
    --save_per_updates {SAVE_EVERY} \
    --last_per_updates {SAVE_EVERY} \
    --epochs {EPOCHS} \
    --finetune \
    --log_samples \
    --logger tensorboard \
    --pretrain /content/base/pretrained_vi.pt

### Khi nào thì dừng

Dataset một giọng, sạch, ~40 phút thường bắt chước được timbre sau khoảng **2 000–6 000 update**. Cứ `SAVE_EVERY` update lại có một checkpoint, nên cách làm đúng là dừng tay và nghe thử, chứ đừng chạy hết `EPOCHS`.

Train quá lâu trên bộ dữ liệu nhỏ sẽ **overfit**: giọng nghe rất giống ở những câu na ná corpus nhưng vỡ khi gặp câu lạ. Nếu thấy dấu hiệu đó thì quay lại checkpoint sớm hơn.

Colab free có thể ngắt phiên giữa chừng — chạy cell 11 để đẩy checkpoint sang Drive định kỳ.

In [ ]:
#@title 10. Nghe thử checkpoint
import glob, os

ckpts = sorted(glob.glob("ckpts/**/*.pt", recursive=True), key=os.path.getmtime)
print("checkpoint có sẵn:")
for c in ckpts:
    print(f"  {os.path.getsize(c)/1e9:.1f} GB  {c}")
CKPT = ckpts[-1] if ckpts else None

REF_WAV = "data/your_dataset/001.wav"
REF_TXT = open("data/your_dataset/001.txt", encoding="utf-8").read().strip()
GEN = "đây là câu hoàn toàn không có trong dữ liệu huấn luyện, dùng để kiểm tra xem giọng có giữ được tự nhiên hay không"

!f5-tts_infer-cli --model F5TTS_Base \
    --ckpt_file {CKPT} \
    --vocab_file /content/base/vocab.txt \
    --ref_audio {REF_WAV} \
    --ref_text "{REF_TXT}" \
    --gen_text "{GEN}" \
    --output_dir /content/out

from IPython.display import Audio, display
for wav in sorted(glob.glob("/content/out/*.wav")):
    print(wav)
    display(Audio(wav))

In [ ]:
#@title 11. Lưu checkpoint sang Google Drive
from google.colab import drive
drive.mount("/content/drive")

import shutil, os, glob
dest = f"/content/drive/MyDrive/omnicast_voice_clone/{VOICE}"
os.makedirs(dest, exist_ok=True)

for c in sorted(glob.glob("ckpts/**/*.pt", recursive=True), key=os.path.getmtime)[-2:]:
    shutil.copy(c, dest)
    print("đã lưu", c)
shutil.copy("/content/base/vocab.txt", dest)
print("xong ->", dest)

### Đưa về OmniCast

Tải checkpoint + `vocab.txt` về máy, rồi suy luận bằng chính repo này ở local (không cần GPU mạnh để infer). Muốn nối vào pipeline thì viết một provider `f5_local` theo khuôn `implementation/src/omnicast/media/providers/tts_chatterbox.py` và đăng ký ở `providers/registry.py`.

Nhắc lại: model sinh ra từ đây mang license **CC-BY-NC-SA-4.0** — chỉ nghiên cứu.